# Step 12. Protein panels, each cohort on its own

Reduces 7,288 probes to a per-cohort panel. weighted gene co-expression network analysis (WGCNA) does the pooling into co-expressed modules; the
only selection steps are a significance test against patient attributes and one stated size
criterion. Nothing from one cohort selects anything in another.

Reads `cohorts/`. Writes `data/run_artifacts/step12_panels.rds`, which steps 13–18 consume.

## The module exclusion criterion

> A module is excluded if it contains ≥10% of the assayed probes (≥729 of 7,288), because its
> eigengene then approximates the first principal component of the entire panel and will correlate
> with clinical variables through overall signal level and technical variation rather than through
> shared mechanism.

The threshold is a judgment, not a derived quantity, placed where that matters least. The
trait-associated module sizes run 16.6%, 8.6%, 6.7%, 5.1%, 2.1%, 1.1%, 0.5%… Any value between
8.6% and 16.6% gives identical results in all nine cohort × condition combinations, so the cut
sits in a gap rather than on top of a module. It excludes exactly two: A's `blue` (16.6%) and C's
`turquoise` (36.4%).

Earlier versions of this pipeline used a pooled-variance panel rule, a panel size of 80, and a
`≤50` module cut. All three were arbitrary numbers presented as data-determined. They are gone.

## Three trait conditions, reported side by side

1. all15, every attribute in `cohorts/clinical-traits.csv`
2. varsel, each cohort's own VarSelLCM selection
3. union, the union of those selections, identical in every cohort

VarSelLCM sees clinical data only, never the proteins, so the selection cannot be circular with
the module–trait test that follows. Note `nbcores = 1`: with more, each worker gets its own RNG
stream and `set.seed()` does not make the selection reproducible.

In [1]:
suppressMessages(library(VarSelLCM))
source("../src/paths.R")
options(stringsAsFactors = FALSE); set.seed(42)


SITES       <- c("A","B","C")
PANEL_TOTAL <- 7288
MAX_FRAC    <- 0.01          # module exclusion: >=1% of assayed probes (>= 73)
MINK_P      <- 2
LINK        <- "ward.D2"
spec        <- read_traits()

build_traits <- function(m, spec){
  out <- data.frame(row.names=rownames(m))
  for (i in seq_len(nrow(spec))){
    v <- m[[spec$source_column[i]]]
    out[[spec$name[i]]] <- switch(spec$type[i],
      numeric = as.numeric(as.character(v)),
      binary  = as.numeric(v=="Positive"),
      ordinal = { lvl<-unique(v[!is.na(v)&v!=""])
                  lvl<-lvl[order(as.numeric(sub("-.*","",lvl)))]
                  as.integer(factor(v, levels=lvl, ordered=TRUE)) },
      stop("unknown trait type ", spec$type[i]))
  }
  out
}

W  <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES
TR <- lapply(SITES, function(s) build_traits(W[[s]]$meta, spec));  names(TR) <- SITES

# ── VarSelLCM per cohort: clinical data only, never the proteins ──────────
varsel_traits <- function(tr){
  clin <- tr
  for (v in spec$name[spec$type=="binary"])
    clin[[v]] <- factor(clin[[v]], levels=c(0,1), labels=c("Neg","Pos"))
  clin <- clin[, vapply(clin, function(x) length(unique(x[!is.na(x)]))>1, logical(1)), drop=FALSE]
  set.seed(42)
  vs <- VarSelCluster(clin, gvals=2:4, vbleSelec=TRUE, crit.varsel="BIC", nbcores=1)
  names(clin)[vs@model@omega==1]
}
VS    <- lapply(TR, varsel_traits)
UNION <- sort(unique(unlist(VS)))
COND  <- list(all15=NULL, varsel=NULL, union=UNION)   # all15/varsel resolved per cohort
traits_for <- function(s, cond)
  switch(cond, all15=spec$name, varsel=VS[[s]], union=UNION)

# ── panel: trait-associated modules, then the 10% exclusion ───────────────
discover <- function(s, cond){
  w  <- W[[s]]; ME <- w$ME[, colnames(w$ME)!="MEgrey", drop=FALSE]
  tt <- traits_for(s, cond)
  tr <- TR[[s]][rownames(ME), tt, drop=FALSE]
  r  <- cor(ME, tr, use="pairwise.complete.obs")
  p  <- 2*pt(-abs(r*sqrt((nrow(ME)-2)/(1-r^2))), nrow(ME)-2)
  q  <- matrix(p.adjust(p,"BH"), nrow=nrow(p), dimnames=dimnames(p))
  sig <- sub("^ME","", rownames(q)[apply(q,1,min) < 0.05])
  sz  <- sapply(sig, function(k) sum(w$mods==k))
  excluded <- sig[sz/PANEL_TOTAL >= MAX_FRAC]
  keep     <- sig[sz/PANEL_TOTAL <  MAX_FRAC]
  list(cohort=s, cond=cond, traits=tt, n_tests=length(q), r=r, q=q,
       sig=sig, keep=keep, excluded=excluded,
       sel=unlist(lapply(keep, function(k) colnames(w$X)[w$mods==k])))
}

D <- list()
for (s in SITES) for (cond in names(COND)) D[[paste(s,cond)]] <- discover(s,cond)

cat("VarSelLCM selections (clinical data only)\n")
for (s in SITES) cat(sprintf("  %s (%d): %s\n", s, length(VS[[s]]), paste(VS[[s]], collapse=", ")))
cat(sprintf("  union (%d): %s\n\n", length(UNION), paste(UNION, collapse=", ")))

cat(sprintf("module exclusion: >= %.0f%% of %d probes (>= %d)\n\n",
            100*MAX_FRAC, PANEL_TOTAL, ceiling(MAX_FRAC*PANEL_TOTAL)))
cat(sprintf("%-6s %-7s %6s %6s %6s %9s %10s  %s\n",
            "cohort","cond","tests","assoc","kept","probes","proteins","excluded"))
for (nm in names(D)){ d <- D[[nm]]
  cat(sprintf("%-6s %-7s %6d %6d %6d %9d %10d  %s\n", d$cohort, d$cond, d$n_tests,
      length(d$sig), length(d$keep), length(d$sel), n_proteins(d$sel),
      if (length(d$excluded)) paste(sprintf("%s(%.1f%%)", d$excluded,
        100*sapply(d$excluded, function(k) sum(W[[d$cohort]]$mods==k))/PANEL_TOTAL), collapse=" ") else "-"))
}

# ── why 1%, and not the 10% an earlier version used ──────────────────────
# A threshold is a choice, so it is reported as a scan rather than asserted.
# For each candidate cut: how well does FREE hierarchical clustering on the
# protein axis recover the WGCNA modules it was never given? Best Jaccard of
# any protein cluster against `ivory` (the interferon module) and `bisque4`
# (the renal one that replicates in all three cohorts).
jaccard <- function(a, b) length(intersect(a, b)) / length(union(a, b))
threshold_scan <- function(s = "A", fracs = c(0.10, 0.05, 0.02, 0.01), ks = 3:8) {
  w <- W[[s]]; sg <- D[[paste(s, "all15")]]$sig
  sz <- sapply(sg, function(k) sum(w$mods == k))
  ref <- list(ivory = colnames(w$X)[w$mods == "ivory"],
              bisque4 = colnames(w$X)[w$mods == "bisque4"])
  do.call(rbind, lapply(fracs, function(f) {
    keep <- names(sz)[sz / PANEL_TOTAL < f]
    sel  <- unlist(lapply(keep, function(k) colnames(w$X)[w$mods == k]))
    Z    <- scale(w$X[, sel, drop = FALSE])
    hc   <- hclust(dist(t(Z), method = "minkowski", p = MINK_P), method = LINK)
    do.call(rbind, lapply(names(ref), function(rn) {
      best <- sapply(ks, function(k) {
        g <- split(names(cutree(hc, k)), cutree(hc, k))
        max(sapply(g, function(x) jaccard(x, ref[[rn]])))
      })
      setNames(data.frame(sprintf("<%.0f%%", 100*f), length(keep), length(sel), rn,
                          t(round(best, 2))),
               c("threshold","modules","probes","module", paste0("k", ks)))
    }))
  }))
}
cat("\nthreshold scan, cohort A -- best Jaccard of a free protein cluster vs the module:\n")
print(threshold_scan(), row.names = FALSE)

saveRDS(list(D = D, VS = VS, UNION = UNION, spec = spec,
             MAX_FRAC = MAX_FRAC, PANEL_TOTAL = PANEL_TOTAL,
             MINK_P = MINK_P, LINK = LINK, SITES = SITES),
        art("step12_panels.rds"))


VarSelLCM selections (clinical data only)


  A (6): SLEDAI_2K, Lymphocyte_count, uPCR, Creatinine, Sm_status, Age_band
  B (6): SLEDAI_2K, C3_level, C4_level, Lymphocyte_count, uPCR, Creatinine
  C (3): SLEDAI_2K, uPCR, Creatinine


  union (8): Age_band, C3_level, C4_level, Creatinine, Lymphocyte_count, SLEDAI_2K, Sm_status, uPCR



module exclusion: >= 1% of 7288 probes (>= 73)



cohort cond     tests  assoc   kept    probes   proteins  excluded


A      all15      780     12      8       164        144  black(2.1%) blue(16.6%) greenyellow(1.1%) yellow(5.1%)
A      varsel     312     10      6        91         73  black(2.1%) blue(16.6%) greenyellow(1.1%) yellow(5.1%)
A      union      416      9      5        83         70  black(2.1%) blue(16.6%) greenyellow(1.1%) yellow(5.1%)
B      all15      345      1      0         0          0  brown(6.7%)
B      varsel     138      1      0         0          0  brown(6.7%)
B      union      184      1      0         0          0  brown(6.7%)
C      all15      690      4      3       119        110  brown(8.6%)
C      varsel     138      5      2       104         97  brown(8.6%) pink(1.1%) turquoise(36.4%)
C      union      368      6      4       153        141  brown(8.6%) pink(1.1%)



threshold scan, cohort A -- best Jaccard of a free protein cluster vs the module:


 threshold modules probes  module   k3   k4   k5   k6   k7   k8
      <10%      11    765   ivory 0.08 0.11 0.11 0.14 0.14 0.14
      <10%      11    765 bisque4 0.05 0.08 0.08 0.10 0.10 0.10
       <5%      10    392   ivory 0.09 0.15 0.39 0.39 0.39 1.00
       <5%      10    392 bisque4 0.07 0.11 0.17 0.17 0.17 0.17
       <2%       9    242   ivory 0.15 0.39 0.39 1.00 1.00 1.00
       <2%       9    242 bisque4 0.11 0.17 0.17 0.17 0.17 1.00
       <1%       8    164   ivory 0.39 0.39 1.00 1.00 1.00 1.00
       <1%       8    164 bisque4 0.17 0.17 0.17 0.17 1.00 1.00


## Reading the panels

**Cohort B has one module under every condition.** Cutting its tests from 345 to 138 does not
change that, so B's single signal is not an artifact of multiple testing, it is what B contains.

**Under `varsel`, cohort C cannot find its interferon module.** VarSelLCM keeps only `SLEDAI_2K`,
`uPCR` and `Creatinine` in C, no autoantibodies, so `mediumpurple3`, whose anti-Sm and
anti-RNP-A associations replicate from cohort A, has no trait left to associate with. Reported
rather than worked around, and the reason the `union` condition exists.

## Probes are not proteins

A SomaScan column is a probe: ISG15 has two, STAT1 has two. Across the menu, 7,288 probes carry
6,399 gene symbols. This matters for the federation rule `p < n` in step 14, where `p` should count
independent measurements.

In [2]:
for (nm in names(D)) {
  d <- D[[nm]]
  if (length(d$sel)) cat(sprintf("%-6s %-7s %s\n", d$cohort, d$cond, size_str(d$sel)))
}

A      all15   164 probes (144 proteins)
A      varsel  91 probes (73 proteins)
A      union   83 probes (70 proteins)
C      all15   119 probes (110 proteins)
C      varsel  104 probes (97 proteins)
C      union   153 probes (141 proteins)


## Where this was run

Rendered notebooks are committed, so each records the machine, the R and the
package versions that produced its output.


In [3]:
run_provenance()

run on   : Annes-MacBook-Pro-193.local ( Darwin 27.0.0 )
date     : 2026-09-25 12:05 EDT 
R        : R version 4.4.3 (2025-02-28) | x86_64-apple-darwin13.4.0 
R comes from: /Users/adeslatt/miniforge3/envs/endotypes-proteomics 
packages :
   WGCNA            1.74
   ComplexHeatmap   2.22.0
   sva              3.54.0
   VarSelLCM        2.1.3.2
   cluster          2.1.8.1
   fpc              2.2.15
   circlize         0.4.18
